# Read the cleaned Parquet file and produce a monthly PDF for departments with embedded matplotlib charts highlighting key graduate student metrics
1. Enrollment trends
2. GPA distribution
3. Financial support
4. Thesis progression
5. Post-degree career outcomes

GIT Commits:
5. Enrollment and financial charts
6. GPA and thesis progression charts
7. career outcomes section
8. HTML report wrapper and auto save

In [1]:
# importing
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from pathlib import Path
from datetime import datetime
import base64
import io
import webbrowser
import tempfile

%matplotlib inline

# Paths

In [2]:
PROJECT_ROOT = Path().resolve().parent

CLEAN_PATH  = PROJECT_ROOT / "data" / "processed" / "grad_students_clean.parquet"
REPORT_DIR  = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_parquet(CLEAN_PATH)
print(f"[LOAD] {len(df):,} rows × {len(df.columns)} columns")
df.head(10)

[LOAD] 550 rows × 41 columns


,student_id,last_name,first_name,academic_term,term_year,program,year_in_program,degree_type,pell_eligible,grant_amt,...,total_financial_support,gpa_risk_flag,thesis_stage_num,degree_program_type,cohort_year,years_since_cohort,pell_x_first_gen,graduated_flag,employed_flag,log_salary
0,1001,FOWLER,Angel,Fall,2020,Phd Chemistry,1,PhD,False,NaN,...,2123.0,True,1,PHD,2020,0,0,False,<NA>,NaN
1,1001,FOWLER,Angel,Spring,2021,Phd Chemistry,1,PhD,False,NaN,...,1865.0,True,1,PHD,2020,1,0,False,<NA>,NaN
2,1001,FOWLER,Angel,Fall,2021,Phd Chemistry,2,PhD,False,NaN,...,21364.0,True,1,PHD,2020,1,0,False,<NA>,NaN
3,1001,FOWLER,Angel,Spring,2022,Phd Chemistry,2,PhD,False,NaN,...,2696.0,True,2,PHD,2020,2,0,False,<NA>,NaN
4,1001,FOWLER,Angel,Fall,2022,Phd Chemistry,3,PhD,False,NaN,...,2172.0,True,2,PHD,2020,2,0,False,<NA>,NaN
5,1001,FOWLER,Angel,Spring,2023,Phd Chemistry,3,PhD,False,NaN,...,1876.0,True,2,PHD,2020,3,0,False,<NA>,NaN
6,1001,FOWLER,Angel,Fall,2023,Phd Chemistry,4,PhD,False,NaN,...,27285.0,True,2,PHD,2020,3,0,False,<NA>,NaN
7,1001,FOWLER,Angel,Spring,2024,Phd Chemistry,4,PhD,False,NaN,...,0.0,True,3,PHD,2020,4,0,False,<NA>,NaN
8,1002,HILL,Jeffrey,Fall,2021,Phd Sociology,1,PhD,False,NaN,...,2002.0,False,1,PHD,2021,0,0,False,<NA>,NaN
9,1002,HILL,Jeffrey,Spring,2022,Phd Sociology,1,PhD,False,NaN,...,23151.0,False,1,PHD,2021,1,0,False,<NA>,NaN


# Styles

In [4]:
PALETTE = {
    "primary":   "#CFB87C",  # CU Gold
    "secondary": "#000000",  # CU Black
    "accent":    "#565A5C",  # Dark Gray
    "danger":    "#9D2235",  # Dark Red (for warnings/highlights)
    "light":     "#F5F5F5",  # Light Gray
    "dark":      "#000000",  # Black
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#F5F5F5",
    "axes.edgecolor": "#CFB87C",      # Gold border
    "axes.grid": True,
    "grid.color": "#D3D3D3",
    "grid.linestyle": "--",
    "grid.alpha": 0.6,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelcolor": "#000000",
    "xtick.color": "#000000",
    "ytick.color": "#000000",
    "text.color": "#000000",
})

# Helper: saving figures base 64 PNG for embedding

In [5]:
# to embed images as text into HTML files for reports
def fig_to_b64(fig) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

def save_figure(fig, filename: str):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [FIG] Saved {path}")

# Chart 1: Enrollment by term and program

In [6]:
def chart_enrollment(df: pd.DataFrame) -> str:
    """
    Interactive enrollment cards — one card per program showing
    total unique students. A term dropdown filters all cards live.
    Returns raw HTML string (not a base64 image).
    """
    # Build term list in chronological order
    tmp = df[["term_label", "term_year", "academic_term"]].drop_duplicates().copy()
    tmp["season"] = tmp["academic_term"].map({"Fall": 1, "Spring": 0})
    terms = tmp.sort_values(["term_year", "season"])["term_label"].tolist()

    # Build enrollment data: {term: {program: n_students}}
    enroll = (
        df.groupby(["term_label", "program"])["student_id"]
        .nunique()
        .reset_index(name="n_students")
    )
    programs = sorted(df["program"].unique().tolist())

    # Serialize to JSON for the JS filter
    import json
    data_json = json.dumps(
        enroll.groupby("term_label")
              .apply(lambda g: dict(zip(g["program"], g["n_students"])))
              .to_dict()
    )
    terms_json  = json.dumps(terms)
    programs_json = json.dumps(programs)

    html = f"""
    <div id="enrollment-section" style="font-family:sans-serif;">

      <div style="margin-bottom:20px;display:flex;align-items:center;gap:16px;">
        <label style="font-weight:bold;color:#000;">Filter by Term:</label>
        <select id="term-select"
                onchange="filterCards()"
                style="padding:8px 14px;border:2px solid #CFB87C;border-radius:6px;
                       font-size:14px;background:white;cursor:pointer;">
          <option value="ALL">All Terms</option>
          {"".join(f'<option value="{t}">{t}</option>' for t in terms)}
        </select>
        <span id="total-badge"
              style="background:#CFB87C;color:#000;padding:6px 14px;
                     border-radius:20px;font-size:13px;font-weight:bold;">
        </span>
      </div>

      <div id="card-grid"
           style="display:grid;grid-template-columns:repeat(auto-fill,minmax(200px,1fr));gap:16px;">
      </div>

    </div>

    <script>
      const DATA     = {data_json};
      const TERMS    = {terms_json};
      const PROGRAMS = {programs_json};

      function getCount(term, program) {{
        if (term === "ALL") {{
          // sum across all terms
          return TERMS.reduce((acc, t) => acc + ((DATA[t] && DATA[t][program]) || 0), 0);
        }}
        return (DATA[term] && DATA[term][program]) || 0;
      }}

      function filterCards() {{
        const term = document.getElementById("term-select").value;
        const grid = document.getElementById("card-grid");
        grid.innerHTML = "";
        let grandTotal = 0;

        PROGRAMS.forEach(program => {{
          const n = getCount(term, program);
          grandTotal += n;
          const card = document.createElement("div");
          card.style.cssText = `
            background: white;
            border-radius: 10px;
            padding: 18px 20px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
            border-left: 5px solid #CFB87C;
            opacity: ${{n === 0 ? 0.35 : 1}};
          `;
          card.innerHTML = `
            <div style="font-size:2rem;font-weight:700;color:#CFB87C;">${{n}}</div>
            <div style="font-size:0.8rem;color:#555;margin-top:6px;line-height:1.4;">
              ${{program}}
            </div>
          `;
          grid.appendChild(card);
        }});

        document.getElementById("total-badge").textContent =
          "Total: " + grandTotal + " student-terms";
      }}

      // Run on load
      filterCards();
    </script>
    """
    return html

# Chart 2 - Financial Support Overview

In [7]:
def chart_financial_support(df: pd.DataFrame):
    """
    Two sub-panels:
      Left  – % Pell-eligible & First-Gen by program (grouped bar)
      Right – Distribution of total financial support (box plot)
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Panel A: Pell & First-Gen rates by program
    summary = (
        df.drop_duplicates("student_id")
        .groupby("program")[["pell_eligible", "first_gen"]]
        .mean()
        .mul(100)
        .round(1)
    )
    x = np.arange(len(summary))
    w = 0.35
    ax1.bar(x - w/2, summary["pell_eligible"], w, label="Pell-Eligible",
            color=PALETTE["primary"], alpha=0.85)
    ax1.bar(x + w/2, summary["first_gen"],      w, label="First-Generation",
            color=PALETTE["secondary"], alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(summary.index, rotation=20, ha="right")
    ax1.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax1.set_title("Pell-Eligible & First-Gen Rate by Program")
    ax1.set_ylabel("% of Students")
    ax1.legend()

    # Panel B: Total financial support distribution
    programs = df["program"].unique()
    data_by_prog = [
        df[df["program"] == p]["total_financial_support"].dropna().values
        for p in programs
    ]
    bp = ax2.boxplot(data_by_prog, patch_artist=True,
                     medianprops={"color": "black", "linewidth": 2})
    colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]]
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax2.set_xticklabels(programs, rotation=20, ha="right")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax2.set_title("Total Financial Support Per Term")
    ax2.set_ylabel("Grant + Stipend ($)")

    fig.tight_layout()
    b64 = fig_to_b64(fig)
    save_figure(fig, "02_financial_support.png")
    return b64

# GPA Trends and AT RISK Flags

In [8]:
def chart_gpa(df: pd.DataFrame):
    """Line chart of mean GPA per term by program + % at-risk overlay."""
    term_gpa = (
        df.groupby(["term_label", "program", "term_year", "academic_term"])["gpa"]
        .mean()
        .reset_index()
    )
    term_gpa["season"] = term_gpa["academic_term"].map({"Fall": 1, "Spring": 0})
    term_gpa = term_gpa.sort_values(["term_year", "season"])

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]]
    for prog, color in zip(df["program"].unique(), colors):
        sub = term_gpa[term_gpa["program"] == prog].drop_duplicates("term_label")
        ax.plot(sub["term_label"], sub["gpa"], marker="o", label=prog, color=color)

    ax.axhline(3.0, color=PALETTE["danger"], linestyle="--", linewidth=1.2,
               label="Probation threshold (3.0)")
    ax.set_title("Mean GPA Trend by Program")
    ax.set_xlabel("Academic Term")
    ax.set_ylabel("Mean GPA")
    ax.set_ylim(2.5, 4.1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    fig.tight_layout()
    b64 = fig_to_b64(fig)
    save_figure(fig, "03_gpa_trend.png")
    return b64
